# RAG Fusion

从 [此 GitHub 仓库](https://github.com/Raudaschl/rag-fusion) 重写，所有功劳归原作者

> RAG-Fusion 是一种搜索方法论，旨在弥合传统搜索范式与人类查询的多方面维度之间的差距。受检索增强生成 (RAG) 功能的启发，该项目更进一步，采用多查询生成和倒数排名融合来重新排名搜索结果。

## 设置

在本示例中，我们将使用 Pinecone 和一些模拟数据。要配置 Pinecone，请设置以下环境变量：

- `PINECONE_API_KEY`: 您的 Pinecone API 密钥

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

In [ ]:
all_documents = {
    "doc1": "Climate change and economic impact.",
    "doc2": "Public health concerns due to climate change.",
    "doc3": "Climate change: A social perspective.",
    "doc4": "Technological solutions to climate change.",
    "doc5": "Policy changes needed to combat climate change.",
    "doc6": "Climate change and its impact on biodiversity.",
    "doc7": "Climate change: The science and models.",
    "doc8": "Global warming: A subset of climate change.",
    "doc9": "How climate change affects daily weather.",
    "doc10": "The history of climate change activism.",
}

In [ ]:
vectorstore = PineconeVectorStore.from_texts(
    list(all_documents.values()), OpenAIEmbeddings(), index_name="rag-fusion"
)

## 定义查询生成器

我们现在将定义一个链来执行查询生成。

In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

In [68]:
from langchain import hub

prompt = hub.pull("langchain-ai/rag-fusion-query-generation")

In [3]:
# prompt = ChatPromptTemplate.from_messages([
#     ("system", "You are a helpful assistant that generates multiple search queries based on a single input query."),
#     ("user", "Generate multiple search queries related to: {original_query}"),
#     ("user", "OUTPUT (4 queries):")
# ])

In [5]:
generate_queries = (
    prompt | ChatOpenAI(temperature=0) | StrOutputParser() | (lambda x: x.split("\n"))
)

## 定义完整链

现在我们可以将所有内容组合起来，定义完整链。此链：

1. 生成大量查询
2. 在检索器中查找每个查询
3. 使用倒数排名融合（reciprocal rank fusion）将所有结果连接起来

请注意，它**不**执行最终的生成步骤。

In [50]:
original_query = "impact of climate change"

In [75]:
vectorstore = PineconeVectorStore.from_existing_index("rag-fusion", OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

In [76]:
from langchain.load import dumps, loads


def reciprocal_rank_fusion(results: list[list], k=60):
    fused_scores = {}
    for docs in results:
        # Assumes the docs are returned in sorted order of relevance
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            previous_score = fused_scores[doc_str]
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    return reranked_results

In [77]:
chain = generate_queries | retriever.map() | reciprocal_rank_fusion

In [78]:
chain.invoke({"original_query": original_query})

[(Document(page_content='Climate change and economic impact.'),
  0.06558258417063283),
 (Document(page_content='Climate change: A social perspective.'),
  0.06400409626216078),
 (Document(page_content='How climate change affects daily weather.'),
  0.04787506400409626),
 (Document(page_content='Climate change and its impact on biodiversity.'),
  0.03306010928961749),
 (Document(page_content='Public health concerns due to climate change.'),
  0.016666666666666666),
 (Document(page_content='Technological solutions to climate change.'),
  0.016666666666666666),
 (Document(page_content='Policy changes needed to combat climate change.'),
  0.01639344262295082)]